# Regression Metrics

**Topic:** Model Evaluation

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox
from IPython.display import display, HTML, clear_output
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout

housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---
## What you'll explore

By the end of this session you will be able to:

- **Interpret** MAE, RMSE, and R² and explain what each one measures in plain English
- **Explain** why MSE amplifies large errors more than MAE — and when that matters for your problem
- **Describe** when R² can be misleading and which backup metric to reach for instead

> **Tip:** Move the prediction slider in the first widget until the prediction error is exactly twice the base value — then watch how MSE responds compared to MAE. That gap is the key insight of this notebook.

---
## How we got here

In `ml_concepts/07_loss_functions.ipynb` you saw how loss functions guide the training process. In `math/calculus/08_the_cost_function.ipynb` you explored the mathematics behind minimizing prediction error. This notebook is the natural next step: once training is done, how do you *report* prediction quality in a way that is meaningful to stakeholders — not just minimized during fitting?

The metrics here are what end up in model cards, dashboards, and deployment decisions.

---
## Why this matters for data science

Regression metrics translate abstract "error" into language that connects to real-world impact. An RMSE of 0.4 on California Housing means predictions are off by $40,000 on average (target is in $100k units). Whether that is acceptable depends entirely on the use case — acceptable for a property tax estimate, catastrophic for a mortgage risk model.

Understanding the math behind each metric also reveals its blind spots. A model with low MAE might still have a few massive errors that RMSE would expose. R² can be high even when the model is terrible if the data has high natural variance.

---
## Try it yourself

In [2]:
out_math = Output()

truth_val = 2.5  # a typical California Housing median value
err_sl = FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1,
    description="Prediction error:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="460px"))

def render_math(change=None):
    error   = err_sl.value
    pred    = truth_val + error
    mae_val = abs(error)
    mse_val = error ** 2
    rmse_val = abs(error)   # sqrt(error^2) = |error| for a single point

    metrics  = ["MAE", "MSE", "RMSE"]
    values   = [mae_val, mse_val, rmse_val]
    formulas = [
        f"mean(|y − ŷ|) = |{error:.2f}| = {mae_val:.3f}",
        f"mean((y − ŷ)²) = ({error:.2f})² = {mse_val:.3f}",
        f"√MSE = √{mse_val:.3f} = {rmse_val:.3f}",
    ]
    colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]]

    traces = [go.Bar(
        x=metrics, y=values,
        marker_color=colors,
        text=[f"{v:.3f}" for v in values],
        textposition="outside",
        customdata=formulas,
        hovertemplate="%{customdata}<extra></extra>",
    )]
    layout = base_layout(
        title=f"Truth = {truth_val}  |  Prediction = {pred:.2f}  |  Error = {error:.2f}",
        xaxis_title="Metric", yaxis_title="Value",
    )
    layout.update(yaxis=dict(range=[0, max(values) * 1.35 + 0.05]), showlegend=False)
    with out_math:
        clear_output(wait=True)
        fig = go.Figure(data=traces, layout=layout)
        display(go.FigureWidget(fig))

err_sl.observe(render_math, names="value")
display(VBox([err_sl, out_math]))
render_math()

In [ ]:
np.random.seed(42)
_rf_w2 = RandomForestRegressor(n_estimators=20, random_state=42)
_rf_w2.fit(X_train, y_train)
_y_pred_w2 = _rf_w2.predict(X_test)
_y_test_w2 = y_test.values

_mae_w2  = mean_absolute_error(_y_test_w2, _y_pred_w2)
_rmse_w2 = np.sqrt(mean_squared_error(_y_test_w2, _y_pred_w2))
_r2_w2   = r2_score(_y_test_w2, _y_pred_w2)
_abs_err_w2 = np.abs(_y_test_w2 - _y_pred_w2)

out_vis = Output()

metric_dd = Dropdown(
    options=["MAE", "RMSE", "R²"],
    value="MAE",
    description="Highlight metric:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="320px"))

def render_vis(change=None):
    metric = metric_dd.value
    metric_vals = {"MAE": _mae_w2, "RMSE": _rmse_w2, "R²": _r2_w2}
    metric_descs = {
        "MAE":  f"MAE = {_mae_w2:.3f} — average absolute error in $100k units",
        "RMSE": f"RMSE = {_rmse_w2:.3f} — penalizes large errors more than MAE",
        "R²":   f"R² = {_r2_w2:.3f} — model explains {_r2_w2*100:.1f}% of variance",
    }
    pmin = float(min(_y_test_w2.min(), _y_pred_w2.min()))
    pmax = float(max(_y_test_w2.max(), _y_pred_w2.max()))
    traces = [
        go.Scatter(x=[pmin, pmax], y=[pmin, pmax], mode="lines",
            line=dict(color=PALETTE["muted"], width=1.5, dash="dash"),
            name="Perfect prediction"),
        go.Scatter(
            x=_y_test_w2, y=_y_pred_w2, mode="markers",
            marker=dict(
                color=_abs_err_w2,
                colorscale=[[0, PALETTE["accent"]], [0.5, PALETTE["primary"]], [1, PALETTE["secondary"]]],
                size=4, opacity=0.6,
                colorbar=dict(title="Abs Error"),
            ),
            name="Test predictions",
        ),
    ]
    layout = base_layout(
        title=f"{metric_descs[metric]}",
        xaxis_title="Actual Median House Value ($100k)",
        yaxis_title="Predicted Median House Value ($100k)",
    )
    layout.update(showlegend=True)
    with out_vis:
        clear_output(wait=True)
        fig = go.Figure(data=traces, layout=layout)
        display(go.FigureWidget(fig))

metric_dd.observe(render_vis, names="value")
display(VBox([metric_dd, out_vis]))
render_vis()

---
## What's happening?

All regression metrics measure prediction error, but they weight errors differently.

**MAE (Mean Absolute Error)** takes the average of absolute differences between predictions and true values. It treats all errors equally — a $10k error counts the same whether it's on a $100k or $900k house. Easy to interpret; robust to outliers.

**MSE (Mean Squared Error)** squares the errors before averaging. Squaring means a $20k error contributes *four times* as much as a $10k error. This makes MSE sensitive to large mistakes. Because it is differentiable everywhere, it is the default loss function during training.

**RMSE (Root Mean Squared Error)** is the square root of MSE — bringing it back to the original units so it is interpretable. It shares MSE's sensitivity to large errors.

**R² (Coefficient of Determination)** measures what fraction of the variance in the target the model explains. An R² of 0.80 means the model explains 80% of the variation in house prices. It ranges from 0 (no better than predicting the mean) to 1 (perfect), and can be negative for truly bad models.

| Metric | Formula | Range | Sensitive to outliers | When to use it |
|---|---|---|---|---|
| MAE | mean(\|y − ŷ\|) | [0, ∞) | No | When large errors are not especially costly |
| MSE | mean((y − ŷ)²) | [0, ∞) | Yes | Training loss; not usually reported directly |
| RMSE | √MSE | [0, ∞) | Yes | When large errors matter more than small ones |
| R² | 1 − SS_res/SS_tot | (−∞, 1] | Moderate | Comparing models across datasets |

---
## Real-world example: Predicting California house prices — how far off is the model?

California Housing is a classic regression benchmark: 20,640 census tracts, 8 features, target is median house value in units of $100k. A Random Forest trained on 80% of the data produces predictions we can evaluate with all four metrics below.

In [4]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import plotly.graph_objects as go
from tkh_utils import PALETTE, FONT, base_layout

np.random.seed(42)
rf = RandomForestRegressor(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

# Predicted vs actual scatter
abs_err = np.abs(y_test.values - y_pred)
layout = base_layout(
    title=f"Random Forest — Predicted vs Actual  (MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f})",
    xaxis_title="Actual Median House Value ($100k)",
    yaxis_title="Predicted Median House Value ($100k)",
)
layout.update(showlegend=True)

fig = go.Figure(layout=layout)
# Perfect prediction line
pmin, pmax = float(y_test.min()), float(y_test.max())
fig.add_trace(go.Scatter(
    x=[pmin, pmax], y=[pmin, pmax],
    mode="lines",
    line=dict(color=PALETTE["muted"], width=1.5, dash="dash"),
    name="Perfect prediction",
))
fig.add_trace(go.Scatter(
    x=y_test.values, y=y_pred,
    mode="markers",
    marker=dict(
        color=abs_err,
        colorscale=[[0, PALETTE["accent"]], [0.5, PALETTE["primary"]], [1, PALETTE["secondary"]]],
        size=4,
        opacity=0.6,
        colorbar=dict(title="Abs Error"),
    ),
    name="Test predictions",
))
fig.show()

---
## Key takeaway

> **MAE gives you the average mistake in your units; RMSE punishes large errors harder; R² tells you how much of the story your model explains — use all three together, never just one.**

---
*Next up: `02_classification_metrics.ipynb` — precision, recall, F1, and the threshold decision that is really a business decision*